VARLIK-ETKİ GRAFİĞİ OLUŞTURULDU

Her özellik kombinasyonlarının verisetinden predict_proba'sı hesaplanır. Hesaplanan predict_proba'ların 1'li, 2'li,..n'li kombinasyonalarının ayrı ayrı regresyonu hesaplanır. Örneğin 36 kombinasyon varsa bizim için 36 kombinasyon için özelliklere ait regresyon katsayısı elde edilir. Biz bu 36 kombinasyonunun özelliklere ait regresyon katsayıları kombinasyondaki eleman sayısına göre  1'li, 2'li,,,,n'li olacak şekilde gruplanır ve her grup için özellik bazında katsayıları toplanarak tek katsayı elde edilir. Böylelikle 7 grubumuz varsa kombinasyonel gruplardan 7 farklı özellik katsayıları elde edilmiş olur ve biz bu verileri zaten grafiklere uyguluyoruz.

PARALELLEŞTİRMELER UYGULUNADI

VARLIK ÖNEMİ VE DEĞER ÖNEMİ KATSAYI DEĞERLERİNİ ELDE EDİYORUZ

onem_katsayi_5MP_8_4_2 (proba katsayısı hesabı) ile onem_katsayi_5MP_8_3_1_1 birleştirildi (normal katsayı hesabı)

Paralleştirmeler Joblib'e dönüştürüldü
Paralelleştirme Windows ile uyumlu hale getirlidi

kendall-tau bazı durumlarda tutarsız sonuçlar verdiği için kendi önerim olan kısmi benzerlikleri de hesaba katacak şekilde Position-Based Overlap Score (POS) kullandım

SHAP için makalede ve tezde kullanılmak üzere grafik çıkarttım.

In [134]:

import pandas as pd
import numpy as np
import time
import psutil

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.utils import resample
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from collections import Counter

import seaborn as sns
import matplotlib.pyplot as plt

import scipy.stats as ss
from itertools import combinations
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import lightgbm as lgb
from catboost import CatBoostClassifier

from sklearn.linear_model import Ridge
from concurrent.futures import ProcessPoolExecutor

import time
from copy import deepcopy

from sklearn.svm import SVC
from sklearn.preprocessing import FunctionTransformer
from vit_tabular_sklearn import make_vit_classifier
import numpy as np

from collections import defaultdict

import numpy as np
from copy import deepcopy

from multiprocessing import Pool
from functools import partial
from multiprocessing import cpu_count
from joblib import Parallel, delayed

In [ ]:
group_size = 7
path = "dataset/adult.csv"


pick_mdl= CatBoostClassifier(
    iterations=200,
    learning_rate=0.01,
    depth=3,
    subsample=0.90,          # OK
    rsm=0.90,                # colsample_bytree karşılığı
    bootstrap_type="Bernoulli",
    loss_function="MultiClass",
    eval_metric="MultiClass",
    random_seed=42,
    verbose=False
)

#pick_mdl = XGBClassifier(n_estimators=220, learning_rate=0.01, max_depth=4, subsample=0.80,  random_state=42)

#pick_mdl = make_vit_classifier(max_epochs=10)


select = None

In [136]:
from copy import deepcopy
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

from typing import Dict, Tuple, List
from fixait import CalcFeatureWeight
from xai_faithfulness_tuner import fiXAItImportanceTuner

group_size=group_size

def prune_p_opt_q_val(
    p_opt: Dict[str, float],
    q_val: Dict[str, float],
    threshold_pct: float = 3.0,
    inclusive: bool = True,   # True => x% and below are eliminated
    drop_zeros: bool = True   # True => 0 and negatives are eliminated
) -> Tuple[Dict[str, float], Dict[str, float], List[str]]:
    """
    Drops are decided ONLY by p_opt:
      1) Drop negatives (and optionally zeros) from p_opt.
      2) For remaining positives in p_opt: compute share = v/sum_pos, drop if share <= threshold.
      3) Apply same drops to BOTH p_opt and q_val (keep them aligned).
      4) Round remaining values to 3 decimals in BOTH dicts.
    """

    p = {k: float(v) for k, v in p_opt.items()}
    q = {k: float(v) for k, v in q_val.items()}

    # 1) non-positive drop based on p_opt only
    if drop_zeros:
        drop = {k for k, v in p.items() if v <= 0.0}
    else:
        drop = {k for k, v in p.items() if v < 0.0}

    # 2) threshold drop by p_opt positive shares (RAW values, no rounding before thresholding)
    pos = {k: v for k, v in p.items() if k not in drop and v > 0.0}
    s = sum(pos.values())
    thr = float(threshold_pct) / 100.0

    if s > 0:
        for k, v in pos.items():
            share = v / s
            if inclusive:
                if share <= thr:
                    drop.add(k)
            else:
                if share < thr:
                    drop.add(k)

    # 3) apply to both and keep aligned keys
    kept = sorted((set(p.keys()) & set(q.keys())) - drop)

    # 4) round to 3 decimals
    p_clean = {k: round(p[k], 3) for k in kept}
    q_clean = {k: round(q[k], 3) for k in kept}

    return p_clean, q_clean, sorted(drop)





# 1) Generate p (existence impact) (plots closed)

cfw = CalcFeatureWeight(
    data_path=path,
    usecols=select,          # ✅ yeni
    model=pick_mdl,
    group_size=group_size,
    n_jobs=-1,
    plot=False,
    test_size=0.15,
    opt_size=0.20,
    stratify=True,
    random_state=42,
    verbose=False
)

p_exist = cfw.new_weight_format
q_val   = cfw.compute_value_impact()

# 2) Buy split+scale all at once.
split = cfw.get_splits()

X_train = pd.DataFrame(split.X_train, columns=split.feature_names)
y_train = pd.Series(split.y_train)

X_opt  = pd.DataFrame(split.X_opt, columns=split.feature_names)
y_opt  = pd.Series(split.y_opt)

X_eval = pd.DataFrame(split.X_test, columns=split.feature_names)
y_eval = pd.Series(split.y_test)

# 3) Fit the model once (with full features)
model_fitted = deepcopy(pick_mdl)
model_fitted.fit(X_train, y_train)

# 4) Tuner: Automatically retrieves splits from CalcFeatureWeight (no PATH, no resplit)
p_opt, faith0, faith1, drop0, drop1 = fiXAItImportanceTuner(
    cfw=cfw,
    model=model_fitted,
    p_exist=p_exist,

    runs_per_feature=30,
    n_jobs=-1,
    prefer="threads",
    verbose=False
)

p, q, dropped_features = prune_p_opt_q_val(
    p_opt=p_opt,
    q_val=q_val,
    threshold_pct=3,
    inclusive=True,
    drop_zeros=True
)


print("fiXAIt calculations were performed and the calculated feature importance values ​​were optimized.")
print("Feature Existence Impact Weights: ", p)
print("Feature Value Impact Weights :", q)


fiXAIt calculations were performed and the calculated feature importance values ​​were optimized.
Feature Existence Impact Weights:  {'age': 0.627, 'capital-gain': 1.0, 'education': 0.3, 'educational-num': 0.459, 'hours-per-week': 0.148, 'marital-status': 0.823}
Feature Value Impact Weights : {'age': 2.759, 'capital-gain': 1.863, 'education': -0.221, 'educational-num': 3.682, 'hours-per-week': 2.488, 'marital-status': -2.039}


In [137]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import accuracy_score, f1_score
from scipy.stats import spearmanr
from sklearn.utils import check_random_state


def evaluate_fidelity(
    model,
    X_train,
    X_test,
    importance_scores,
    top_k=7,
    metric='accuracy',
    max_depth='auto',
    random_state=42,
    verbose=False,
    epsilon=1e-3,
    normalize=False,
):
    # 1) Get the importance values ​​into Series and index them.
    imp = pd.Series(importance_scores, index=X_train.columns,
                    name='Importance', dtype=float)
    if normalize:
        max_abs = imp.abs().max()
        if max_abs > 0:
            imp = imp / max_abs

    # 2) Filtring
    imp_abs = imp.abs()
    imp_filtered = imp_abs[imp_abs > epsilon]
    if imp_filtered.empty:
        if verbose:
            print(f"No features above |epsilon|={epsilon}, falling back.")
        imp_filtered = imp_abs

    # 3) Top-k
    top_feats = imp_filtered.sort_values(ascending=False)\
                             .head(top_k).index.tolist()
    if verbose:
        print("Features used:", top_feats)

    # 4) Black-box predictions
    y_bb_train = model.predict(X_train)
    y_bb_test  = model.predict(X_test)

    # 5) Select depth
    if max_depth == 'auto':
        param_grid = {'max_depth': list(range(2, 11))}
        cv = StratifiedKFold(n_splits=5, shuffle=True,
                             random_state=random_state)
        gs = GridSearchCV(
            DecisionTreeClassifier(random_state=random_state),
            param_grid,
            scoring='accuracy',
            cv=cv,
            n_jobs=-1
        )
        gs.fit(X_train[top_feats], y_bb_train)
        best_depth = gs.best_params_['max_depth']
    else:
        best_depth = max_depth
    if verbose:
        print(f"Chosen max_depth: {best_depth}")

    # 6) Last surrogate
    surrogate = DecisionTreeClassifier(max_depth=best_depth, random_state=random_state)
    surrogate.fit(X_train[top_feats], y_bb_train)

    # 7) Fidelity
    y_surr = surrogate.predict(X_test[top_feats])
    if metric == 'accuracy':
        fidelity = accuracy_score(y_bb_test, y_surr)
    else:
        fidelity = f1_score(y_bb_test, y_surr, average='weighted')
    if verbose:
        print(f"Fidelity: {fidelity:.4f}")

    return fidelity, surrogate


def evaluate_faithfulness(
    model,
    X_train: pd.DataFrame,
    y_train: pd.Series,
    X_test: pd.DataFrame,
    y_test: pd.Series,
    importance_scores: dict,
    *,
    # --- legacy arguments ---
    top_k: int = 7,
    metric: str = "accuracy",
    random_state: int = 42,
    runs_per_feature: int = 30,
    epsilon: float = 1e-8,
    normalize: bool = False,
    verbose: bool = False,
    grid_resolution: int = 30,
    pdp_percentiles: tuple = (0.05, 0.95),
    # --- new arguments ---
    drop_mode: str = "metric",       # "metric" | "prob"
    class_id: int | None = None,     # for drop_mode="prob" (probability column index)
    abs_drop: bool = True,           # take |.| of the probability drop
    conditional_permutation: bool = False,  # more robust to correlation
    # --- practical: optional PD-variance computation ---
    compute_pd_var: bool = False,
    # --- parallel ---
    n_jobs: int = 1,
    prefer: str = "threads",
):
    """
Compute feature faithfulness using either:
  - drop in a classic performance metric (accuracy/f1/log_loss), or
  - drop in the predicted class probability.

Returns
-------
faithfulness : float
drop_impacts : dict[str, float]
(optional) pd_var : dict[str, float]
    Only returned when compute_pd_var=True.
"""

    # ------- 1) Importance scores + filtering ---------------------------- #
    imp = pd.Series(importance_scores, dtype=float)
    if normalize:
        m = imp.abs().max()
        if m > 0:
            imp /= m

    active = imp.abs() > epsilon
    feats = (
        imp.loc[active].abs().nlargest(top_k).index.tolist()
        if active.any() and 0 < top_k < active.sum()
        else imp.loc[active].index.tolist() if active.any()
        else imp.index.tolist()
    )
    if verbose:
        print(f"Selected {len(feats)} features:", ", ".join(feats))

    # ------- 2) Baseline score / probability ---------------------------- #
    metric = metric.lower()
    if drop_mode == "metric":
        if metric == "accuracy":
            scorer = accuracy_score
        elif metric == "f1":
            scorer = lambda y, yp: f1_score(y, yp, average="weighted")
        elif metric == "log_loss":
            if not hasattr(model, "predict_proba"):
                raise ValueError("metric='log_loss' requires model.predict_proba.")
            base_score = -log_loss(y_test, model.predict_proba(X_test))  # higher = better
            scorer = None
        else:
            raise ValueError(f"Unsupported metric: {metric}")

        if metric != "log_loss":
            y_pred_base = model.predict(X_test)
            base_score = scorer(y_test, y_pred_base)

    elif drop_mode == "prob":
        if not hasattr(model, "predict_proba"):
            raise ValueError("drop_mode='prob' requires model.predict_proba.")
        if class_id is None:
            class_id_vec = y_test.to_numpy()
            proba = model.predict_proba(X_test)
            base_scores = proba[np.arange(len(X_test)), class_id_vec]
        else:
            base_scores = model.predict_proba(X_test)[:, class_id]
    else:
        raise ValueError("drop_mode must be 'metric' or 'prob'.")

    if verbose and drop_mode == "metric":
        print(f"Base {metric}: {base_score:.4f}")

    # ------- 3) Impact via permutation ---------------------------------- #
    rng_master = check_random_state(random_state)

    # For conditional_permutation: precompute bin indices once per feature
    groups_map: Dict[str, List[np.ndarray]] = {}
    if conditional_permutation:
        for f in feats:
            q = pd.qcut(X_test[f], q=10, duplicates="drop")
            groups: List[np.ndarray] = []
            for cat in q.unique():
                idx = np.where(q == cat)[0]
                if len(idx) > 1:
                    groups.append(idx)
            groups_map[f] = groups

    # Pre-generate per-run seeds for deterministic parallel execution
    seeds_map: Dict[str, np.ndarray] = {
        f: rng_master.randint(0, np.iinfo(np.int32).max, size=runs_per_feature)
        for f in feats
    }

    def _impact_one_run(f: str, seed: int) -> float:
        rng = np.random.RandomState(int(seed))
        Xp = X_test.copy()
        col = Xp[f].to_numpy().copy()

        if conditional_permutation:
            for idx in groups_map.get(f, []):
                rng.shuffle(col[idx])
        else:
            rng.shuffle(col)

        Xp[f] = col

        if drop_mode == "metric":
            if metric == "log_loss":
                score = -log_loss(y_test, model.predict_proba(Xp))
            else:
                score = scorer(y_test, model.predict(Xp))
            return float(base_score - score)

        # "prob"
        proba = model.predict_proba(Xp)
        if class_id is None:
            p = proba[np.arange(len(Xp)), class_id_vec]
        else:
            p = proba[:, class_id]

        diff = base_scores - p
        if abs_drop:
            diff = np.abs(diff)
        else:
            diff = np.maximum(diff, 0)
        return float(np.mean(diff))

    def _impact_for_feature(f: str) -> Tuple[str, float]:
        impacts = [_impact_one_run(f, s) for s in seeds_map[f]]
        return f, float(np.mean(impacts))

    if (n_jobs is None) or (int(n_jobs) == 1) or (len(feats) <= 1):
        drop_impacts = dict(_impact_for_feature(f) for f in feats)
    else:
        results = Parallel(n_jobs=n_jobs, prefer=prefer)(
            delayed(_impact_for_feature)(f) for f in feats
        )
        drop_impacts = dict(results)

    if verbose:
        for f in feats:
            print(f"Impact {f}: {drop_impacts[f]:.4f}")

    # ------- 4) (Optional) PD variance ---------------------------------- #
    pd_var: Dict[str, float] = {}
    if compute_pd_var:
        # Note: PD variance is NOT included in faithfulness. It is only diagnostic.
        target_col = None
        # Caution: predict_proba(...).mean(axis=1) can become uninformative in multiclass.
        # Here we track class_id if provided; otherwise we track the true-class column.
        if hasattr(model, "predict_proba"):
            if (drop_mode == "prob") and (class_id is not None):
                target_col = class_id
            else:
                # true-class column (assumes label == column index) – consistent with the previous setup
                target_col = None

        low_p, high_p = pdp_percentiles

        def _pd_var_for_feature(f: str) -> Tuple[str, float]:
            low, high = X_train[f].quantile(low_p), X_train[f].quantile(high_p)
            grid = np.linspace(low, high, grid_resolution)

            pd_mat = np.zeros((len(X_test), len(grid)))
            for i, v in enumerate(grid):
                Xt = X_test.copy()
                Xt[f] = v
                if hasattr(model, "predict_proba"):
                    proba = model.predict_proba(Xt)
                    if target_col is not None:
                        pr = proba[:, target_col]
                    else:
                        cls = y_test.to_numpy()
                        pr = proba[np.arange(len(Xt)), cls]
                else:
                    pr = model.predict(Xt)
                pd_mat[:, i] = pr

            return f, float(np.mean(np.var(pd_mat, axis=1)))

        if (n_jobs is None) or (int(n_jobs) == 1) or (len(feats) <= 1):
            pd_var = dict(_pd_var_for_feature(f) for f in feats)
        else:
            results = Parallel(n_jobs=n_jobs, prefer=prefer)(
                delayed(_pd_var_for_feature)(f) for f in feats
            )
            pd_var = dict(results)

        if verbose:
            for f in feats:
                print(f"PD-var {f}: {pd_var[f]:.4f}")

        # ------- 5) Spearman correlation ------------------------------------ #
    imp_vals = [imp.abs()[f] for f in feats]
    impact_vals = [drop_impacts[f] for f in feats]

    if len(set(imp_vals)) <= 1 or len(set(impact_vals)) <= 1:
        faithfulness = 0.0
        if verbose:
            print("No variance → faithfulness = 0")
    else:
        faithfulness, _ = spearmanr(imp_vals, impact_vals)
        if verbose:
            print(f"Faithfulness (Spearman): {faithfulness:.4f}")

    if compute_pd_var:
        return faithfulness, drop_impacts, pd_var
    return faithfulness, drop_impacts

t =7
if __name__ == '__main__':
    # Example usage

    split = cfw.get_splits()

    X_train = pd.DataFrame(split.X_train, columns=split.feature_names)
    y_train = pd.Series(split.y_train)

    # Eğer tuner optimize için X_opt kullandıysa:
    X_opt  = pd.DataFrame(split.X_opt, columns=split.feature_names)
    y_opt  = pd.Series(split.y_opt)

    # For the final report, X_eval:
    X_test = pd.DataFrame(split.X_test, columns=split.feature_names)  # eval/test
    y_test = pd.Series(split.y_test)

    bb_model = deepcopy(pick_mdl)
    bb_model.fit(X_train, y_train)

    fidel_fixait, _ = evaluate_fidelity(
        model=bb_model,
        X_train=X_train,
        X_test=X_test,
        importance_scores=p,
        top_k=t,
        metric='accuracy',
        max_depth='auto',
        verbose=False
    )
    faith_fixait, impacts = evaluate_faithfulness( #
        model=bb_model,
        X_train=X_train,
        y_train=y_train,
        X_test=X_test,
        y_test=y_test,
        importance_scores=p,
        top_k=t,
        metric='accuracy',
        verbose=False
    )

print(f'\nfiXAIt → Fidelity: {fidel_fixait:.4f} | Faithfulness: {faith_fixait:.4f}')
print("Self Consistency: ", cfw.alg_consistency)


fiXAIt → Fidelity: 0.9913 | Faithfulness: 0.9429
Self Consistency:  0.816
